In [1]:
import pandas as pd

In [2]:
file_path = "../data/marketing_customer_purchase.csv"
# Read the CSV file and load it into a pandas DataFrame
df = pd.read_csv(file_path)

In [3]:
print(df.head())
print(df.shape)
print(df.columns)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().sum())

   Age  Gender       City   Income  Website_Visits  Time_on_Site_Minutes  \
0   56    Male    Calgary  47096.0               4                  5.99   
1   69    Male     Ottawa  88056.0               3                  4.52   
2   46    Male     Ottawa  40944.0               9                  8.04   
3   32    Male    Toronto  61136.0               6                  8.78   
4   60  Female  Vancouver  65927.0               6                  6.17   

   Previous_Purchases Membership_Level Email_Opened Discount_Used  \
0                   4             Gold          Yes            No   
1                   3          Premium          Yes           Yes   
2                   3             Gold           No            No   
3                   2          Premium           No           Yes   
4                   3           Silver           No            No   

   Social_Media_Clicks Cart_Abandoned Purchased  
0                    1             No       Yes  
1                    5      

In [4]:
print("Number of duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Number of duplicate rows after cleaning:", df.duplicated().sum())


Number of duplicate rows before cleaning: 10
Number of duplicate rows after cleaning: 0


In [5]:
# Check missing values
print(df.isnull().sum())
# Fill missing numerical values with the median
# Median is often safer than mean when the column has extreme values.
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Income"] = df["Income"].fillna(df["Income"].median())
df["Website_Visits"] = df["Website_Visits"].fillna(df["Website_Visits"].median())
df["Time_on_Site_Minutes "] = df["Time_on_Site_Minutes"].fillna(df["Time_on_Site_Minutes"].median())
df["City"] = df["City"].fillna("Unknown")

Age                       0
Gender                    0
City                      0
Income                   25
Website_Visits            0
Time_on_Site_Minutes     25
Previous_Purchases        0
Membership_Level        289
Email_Opened             25
Discount_Used             0
Social_Media_Clicks       0
Cart_Abandoned            0
Purchased                 0
dtype: int64


In [6]:
text_columns = ['Age', 'Gender', 'City', 'Income', 'Website_Visits',
       'Time_on_Site_Minutes', 'Previous_Purchases', 'Membership_Level',
       'Email_Opened', 'Discount_Used', 'Social_Media_Clicks',
       'Cart_Abandoned', 'Purchased']
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()
# Example: make city names consistent
# df["City"] = df["City"].str.title()

In [7]:
feature_columns = [
    "Age",
    "Gender",
    "City",
    "Income",
    "Website_Visits",
    "Time_on_Site_Minutes",
    "Previous_Purchases",
    "Membership_Level",
    "Email_Opened",
    "Discount_Used",
    "Social_Media_Clicks",
    "Cart_Abandoned"
]

X = df[feature_columns]
y = df["Purchased"].str.strip().map({"No": 0, "Yes": 1})
print(X.head())
print(y.head())

  Age  Gender       City   Income Website_Visits Time_on_Site_Minutes  \
0  56    Male    Calgary  47096.0              4                 5.99   
1  69    Male     Ottawa  88056.0              3                 4.52   
2  46    Male     Ottawa  40944.0              9                 8.04   
3  32    Male    Toronto  61136.0              6                 8.78   
4  60  Female  Vancouver  65927.0              6                 6.17   

  Previous_Purchases Membership_Level Email_Opened Discount_Used  \
0                  4             Gold          Yes            No   
1                  3          Premium          Yes           Yes   
2                  3             Gold           No            No   
3                  2          Premium           No           Yes   
4                  3           Silver           No            No   

  Social_Media_Clicks Cart_Abandoned  
0                   1             No  
1                   5             No  
2                   6            Ye

In [8]:
# Step 5: Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42,
stratify=y
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (640, 12)
X_test shape: (160, 12)
y_train shape: (640,)
y_test shape: (160,)


In [9]:
# Step 6: Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numerical_features = [
    "Age",
    "Income",
    "Website_Visits",
    "Time_on_Site_Minutes",
    "Previous_Purchases",
    "Social_Media_Clicks"
]

categorical_features = [
    "Gender",
    "City",
    "Membership_Level",
    "Email_Opened",
    "Discount_Used",
    "Cart_Abandoned"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


Processed training shape: (640, 24)
Processed test shape: (160, 24)


In [10]:
# Step 7: Train a baseline model
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(X_train_processed, y_train)

y_pred = baseline_model.predict(X_test_processed)

print(y_pred)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1]


In [11]:
# Cleaner workflow: preprocessing + model in one Pipeline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
clf = Pipeline(steps=[
("preprocessor", preprocessor),
("model", LogisticRegression(max_iter=1000))
])
# Train the full pipeline on raw X_train
clf.fit(X_train, y_train)
# Predict using raw X_test; preprocessing is applied automatically
y_pred = clf.predict(X_test)
print(y_pred)


[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1]


In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.76875
Confusion Matrix:
[[  1  35]
 [  2 122]]
Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.03      0.05        36
           1       0.78      0.98      0.87       124

    accuracy                           0.77       160
   macro avg       0.56      0.51      0.46       160
weighted avg       0.68      0.77      0.68       160

